# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** **One row = One pseudonymized content item (`content_id`)** belonging to a pseudonymized client (`client_id`).

**Time Window:**
* **Starter Dataset (`data/raw/content_refresh_anonymized.csv`):** Fixed trailing 90-day snapshot of search performance and engagement metrics.
* **Warehouse Release (`hf://datasets/FlyRank/internship-warehouse`):** Daily panel data spanning Jan 27, 2025 to June 30, 2026 (~17 months). For feature engineering and model training, we isolate mid-panel month `month=2026-03` to prevent outcome window overlap with the sealed test month (`month=2026-06`).

In [1]:
# Code check: Verify grain uniqueness and missing/duplicate check
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_rows = len(df)
unique_content = df['content_id'].nunique()
dup_rows = df.groupby('content_id').filter(lambda x: len(x) > 1).shape[0]

print("=== GRAIN VERIFICATION ===")
print(f"Total Rows: {total_rows:,}")
print(f"Unique Content IDs: {unique_content:,}")
print(f"Duplicate Rows at Grain: {dup_rows}")
assert total_rows == unique_content, 'Grain violation: content_id is not unique!'

=== GRAIN VERIFICATION ===
Total Rows: 30,000
Unique Content IDs: 30,000
Duplicate Rows at Grain: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Field Name(s) | Description / Rules |
|---|---|---|
| **Feature** | `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `engaged_sessions_90d`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `has_word_count`, `has_search_volume`, `has_pos_data` | Historical metrics knowable strictly BEFORE the prediction moment. |
| **Label / Proxy** | `is_declining_label` (`trend_direction == 'down'`) | Binary target outcome indicating performance decline. |
| **Context** | `content_id`, `client_id`, `content_type`, `main_intent`, `impression_tier`, `position_tier`, `freshness_tier` | Used for grouped splits, panel joins, and subgroup analysis — NEVER direct features. |
| **Excluded** | `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, `impressions_prev_30d`, `clicks_prev_30d` | **Target derivations / Sub-window components:** Directly used to compute the label (causes 100% target leakage). |

In [2]:
# Code check: Verify field buckets and check for zero overlap between features and target
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 
            'clicks_90d', 'pageviews_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 
            'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']
target = ['is_declining_label']
excluded = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']

print(f"Features count: {len(features)}")
print(f"Target count: {len(target)}")
print(f"Excluded count: {len(excluded)}")
assert len(set(features).intersection(set(excluded))) == 0, 'Leakage error: Excluded columns in features!'

Features count: 15
Target count: 1
Excluded count: 6


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Empirical Verification Claims:
1. **Grain Uniqueness:** `content_id` uniquely identifies each row (0 duplicates).
2. **Patterned Missingness:** Missingness follows `content_type` structure: `feedly article` has **100% missing keyword data** (`search_volume`, `cpc`), whereas `keyword article` has **28.3% missing word count**.
3. **Rate Column Gotchas:** `avg_position == 0` represents 1,205 rows with "no position data", not rank zero.

In [3]:
# Code check: Verify patterned missingness and gotcha flags
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Patterned missingness by content_type
missing_by_type = df.groupby('content_type')[['search_volume', 'word_count', 'cpc']].apply(lambda g: g.isna().mean())
print("=== PATTERNED MISSINGNESS BY CONTENT TYPE ===")
print(missing_by_type)

# 2. Rate column gotcha: avg_position == 0
no_pos_count = (df['avg_position'] == 0).sum()
print(f"\nNo Position Data (avg_position == 0): {no_pos_count:,} rows ({no_pos_count/len(df):.2%})")

=== PATTERNED MISSINGNESS BY CONTENT TYPE ===
                    search_volume  word_count       cpc
content_type                                           
comparison article       0.000000    0.000000  0.000000
feedly article           1.000000    0.000000  1.000000
keyword article          0.013673    0.282979  0.013673

No Position Data (avg_position == 0): 1,205 rows (4.02%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.